# Interactive Visualization of Cis-Regulatory Elements (PlantCARE)

In this notebook, motif occurrences are loaded into pandas DataFrames and plotted **directly with Seaborn ( / )**, allowing direct control over figure aesthetics and plotting parameters.

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Visual configuration
%matplotlib inline
plt.rcParams["figure.dpi"] = 150
plt.rcParams["font.family"] = "sans-serif"

## 1. Load Count Data
Using the unified non-overlapping dataset ().

In [ ]:
results_dir = os.path.abspath("../results")

# Load count matrices indexed by Gene
df_vper = pd.read_csv(os.path.join(results_dir, "Vper_motif_counts_merged.tsv"), sep="	", index_col="Gene")
df_vtub = pd.read_csv(os.path.join(results_dir, "Vtub_motif_counts_merged.tsv"), sep="	", index_col="Gene")

# Drop ubiquitous core elements (TATA/CAAT-box) and parsing artifacts
drop_cols = ["TATA-box", "CAAT-box", "TATA"] + [c for c in df_vper.columns if "Unnamed" in c]
df_vper_clean = df_vper.drop(columns=drop_cols, errors="ignore")
df_vtub_clean = df_vtub.drop(columns=drop_cols, errors="ignore")

# Select biologically relevant motifs for concise display
selected_motifs = [
    "ABRE", "CGTCA-motif", "TGACG-motif", "TCA-element",      # Phytohormone / ABA
    "MBS", "LTR", "ARE", "TC-rich repeats", "W box",       # Abiotic Stress
    "MYC", "MYB", "RY-element"                              # Development / Transcription Factors
]

# Filter subsets
vper_sub = df_vper_clean[[m for m in selected_motifs if m in df_vper_clean.columns]]
vtub_sub = df_vtub_clean[[m for m in selected_motifs if m in df_vtub_clean.columns]]

vper_sub.head()

## 2. Direct Heatmap with  (*Vellozia peripherica*)

In [ ]:
plt.figure(figsize=(10, 6))

sns.heatmap(
    vper_sub,
    cmap="YlOrRd",
    annot=True,          # Show counts inside cells
    fmt="d",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Occurrences (non-overlapping sites)"}
)

plt.title("Vellozia peripherica — OSCA Promoter Cis-Regulatory Elements", fontsize=13, pad=12)
plt.xlabel("Cis-Regulatory Element")
plt.ylabel("OSCA Gene")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 3. Direct Heatmap with  (*Vellozia tubiflora*)

In [ ]:
plt.figure(figsize=(10, 6))

sns.heatmap(
    vtub_sub,
    cmap="YlOrRd",
    annot=True,
    fmt="d",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Occurrences (non-overlapping sites)"}
)

plt.title("Vellozia tubiflora — OSCA Promoter Cis-Regulatory Elements", fontsize=13, pad=12)
plt.xlabel("Cis-Regulatory Element")
plt.ylabel("OSCA Gene")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 4. Side-by-Side Comparison ( + )
Shared colorbar scale allows direct cross-species comparison.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)

# Shared vmax across both species
vmax_val = max(vper_sub.values.max(), vtub_sub.values.max())

# Left Panel: Vper (Evergreen)
sns.heatmap(
    vper_sub,
    ax=axes[0],
    cmap="YlGnBu",
    annot=True,
    fmt="d",
    vmin=0,
    vmax=vmax_val,
    linewidths=0.5,
    cbar=False
)
axes[0].set_title("Vellozia peripherica (Evergreen / Drought-Avoiding)", fontsize=12, fontweight="bold")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha="right")
axes[0].set_xlabel("Cis-Regulatory Element")
axes[0].set_ylabel("OSCA Gene")

# Right Panel: Vtub (Resurrection)
sns.heatmap(
    vtub_sub,
    ax=axes[1],
    cmap="YlGnBu",
    annot=True,
    fmt="d",
    vmin=0,
    vmax=vmax_val,
    linewidths=0.5,
    cbar_kws={"label": "Non-Overlapping Site Occurrences"}
)
axes[1].set_title("Vellozia tubiflora (Resurrection / Desiccation-Tolerant)", fontsize=12, fontweight="bold")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha="right")
axes[1].set_xlabel("Cis-Regulatory Element")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

## 5. Hierarchical Clustering ()
Groups OSCA loci by regulatory profile similarity.

In [ ]:
g = sns.clustermap(
    vper_sub,
    cmap="viridis",
    annot=True,
    fmt="d",
    figsize=(11, 7),
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Occurrences"}
)

g.fig.suptitle("Hierarchical Clustering of OSCA Promoters — Vellozia peripherica", y=1.03, fontsize=14)
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha="right")
plt.show()